In [ ]:
    ############    #############   Multiprocessing, threading, queues, semaphores   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.1 Production Python
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   Queues and Semaphores   #############   ##############   

 =>  A Queue is the standard way to hand work safely between producers and consumer
       workers (threads, processes, or asyncio tasks) without shared-state races.

 =>  A Semaphore bounds how many coroutines/threads may enter a critical section at once,
       e.g. 'never more than 5 concurrent calls to the LLM provider'.

 =>  This is exactly how you protect a downstream dependency (DB, external API, GPU) from
       being overwhelmed by a burst of concurrent requests.


In [ ]:
import asyncio
import random

MAX_CONCURRENT_CALLS = 3
semaphore = asyncio.Semaphore(MAX_CONCURRENT_CALLS)

async def call_llm(request_id: int) -> None:
    async with semaphore:
        print(f"request {request_id}: calling model...")
        await asyncio.sleep(random.uniform(0.2, 0.5))
        print(f"request {request_id}: done")

async def main():
    await asyncio.gather(*(call_llm(i) for i in range(10)))

await main()


In [ ]:
 =>  Even though 10 requests arrive at once, at most 3 are ever "in flight" against the
       (simulated) model provider at the same time -- the rest wait their turn at the
       semaphore.

 =>  In production this is how you cap concurrent calls to a rate-limited or expensive
       backend (an LLM API, a GPU inference server, a connection pool -- see the Connection
       Pooling notebook).


In [ ]:
import queue
import threading

work_queue: queue.Queue = queue.Queue()
for i in range(5):
    work_queue.put(f"job-{i}")

def worker(worker_id: int):
    while True:
        try:
            job = work_queue.get(timeout=0.5)
        except queue.Empty:
            return
        print(f"worker {worker_id} processing {job}")
        work_queue.task_done()

threads = [threading.Thread(target=worker, args=(i,)) for i in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("all jobs processed")


In [ ]:
 =>  A thread-safe queue.Queue lets multiple worker threads pull jobs safely without any
       manual locking -- the queue itself handles the synchronization.

 =>  multiprocessing.Queue works the same way across process boundaries (with the extra
       cost of pickling data to send it between processes) -- reach for it when your workers
       are separate processes, not threads.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Rewrite the queue example using multiprocessing.Queue and multiprocessing.Process
           instead of threading, and confirm it still works.

 =>  [ ] Add a semaphore in front of a real outbound call in one of your own scripts, and
           verify (with prints/logging) that concurrency is actually capped.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Forgetting task_done() -- queue.join() (if used) will hang forever waiting for
       acknowledgements that never come.

 =>  Sizing a semaphore/pool by guesswork instead of the downstream service's actual
       documented concurrency limit -- too high still overwhelms it, too low leaves
       throughput on the table.
